# Data Preparation

## Tokenizer

In [1]:
import pickle
import torch

class WubiCharTokenizer:
    """
    Tokenizes Chinese text into sequences of atomic Wubi symbols (letters + separator).
    Each character becomes a fixed number of tokens (its Wubi code letters + separator).
    No merging across characters occurs.
    """
    def __init__(self, wubi_dict_path):
        with open(wubi_dict_path, 'rb') as f:
            self.ch2wubi = pickle.load(f)

        # Build vocabulary: all possible letters + separator
        self.sep = '#'
        letters = set()
        for code in self.ch2wubi.values():
            letters.update(code)
        # Also include digits? Original mapping strips digits, so just letters.
        self.vocab = ['[PAD]', '[UNK]', self.sep] + sorted(letters)
        self.stoi = {s: i for i, s in enumerate(self.vocab)}
        self.itos = {i: s for s, i in self.stoi.items()}

    def encode(self, text):
        """
        Convert text to a list of token IDs and a list of character spans.
        Returns:
            ids: list of int (token IDs)
            char_spans: list of (start, end) token indices for each character
        """
        text = text.lower()
        ids = []
        char_spans = []
        for ch in text:
            # Get Wubi code (fallback to character itself)
            code = self.ch2wubi.get(ch, ch)
            # Convert each symbol in the code to an ID
            code_ids = [self.stoi.get(c, self.stoi['[UNK]']) for c in code]
            start = len(ids)
            ids.extend(code_ids)
            ids.append(self.stoi[self.sep])
            end = len(ids)
            char_spans.append((start, end))
        return ids, char_spans

    def decode(self, ids):
        """For debugging: convert IDs back to string."""
        return ''.join(self.itos[i] for i in ids if i != self.stoi[self.sep])

In [2]:
# Load the Wubi mapping (from the SubCharTokenization repo)
wubi_dict = "/kaggle/input/notebooks/davidvista/wubi-tokenizer/chinese_to_wubi.pkl"
tokenizer = WubiCharTokenizer(wubi_dict)

text = "我是最高的学生"
ids, spans = tokenizer.encode(text)

print("Token IDs:", ids)
print("Character spans:", spans)
# Output: spans like [(0,5), (5,8), (8,15), ...] depending on code lengths

Token IDs: [29, 2, 22, 2, 22, 14, 2, 37, 25, 2, 30, 2, 21, 28, 2, 32, 19, 2]
Character spans: [(0, 2), (2, 4), (4, 7), (7, 10), (10, 12), (12, 15), (15, 18)]


In [3]:
# Inverse mapping from Wubi to Character
with open("/kaggle/input/notebooks/davidvista/wubi-tokenizer/wubi_to_chinese.pkl", 'rb') as f:
    wubi2ch = pickle.load(f)

## Pinyin Mapping

In [4]:
INITIALS = [
    "", "b","p","m","f","d","t","n","l","g","k","h",
    "j","q","x","zh","ch","sh","r","z","c","s"
]

FINALS = [
    "a","ai","an","ang","ao","e","ei","en","eng","er",
    "i","ia","ian","iang","iao","ie","in","ing","iong","iu",
    "o","ong","ou",
    "u","ua","uai","uan","uang","ui","un","uo",
    "v","ve","van","vn"
]

TONES = ["1","2","3","4","5"]

ZERO_INITIAL_MAP = {
    # y-series
    "yi":"i",
    "ya":"ia",
    "yao":"iao",
    "ye":"ie",
    "you":"iu",
    "yan":"ian",
    "yin":"in",
    "yang":"iang",
    "ying":"ing",
    "yong":"iong",

    "yu":"v",
    "yue":"ve",
    "yuan":"van",
    "yun":"vn",

    # w-series
    "wu":"u",
    "wa":"ua",
    "wo":"uo",
    "wai":"uai",
    "wei":"ui",
    "wan":"uan",
    "wen":"un",
    "wang":"uang",
    "weng":"ong",
}


def normalize_zero_initial(base):
    return ZERO_INITIAL_MAP.get(base, base)


def normalize_jqx(final):
    if final.startswith("u"):
        mapping = {
            "u": "v",
            "ue": "ve",
            "uan": "van",
            "un": "vn",
        }
        return mapping.get(final, final)
    return final


init2idx = {v:i for i,v in enumerate(INITIALS)}
final2idx = {v:i for i,v in enumerate(FINALS)}
tone2idx = {v:i for i,v in enumerate(TONES)}

idx2init = {i:v for v,i in init2idx.items()}
idx2final = {i:v for v,i in final2idx.items()}
idx2tone = {i:v for v,i in tone2idx.items()}


def pinyin_to_idx(token: str):
    tone = token[-1]

    if not tone.isdigit():
        tone = "5" # neutral tone, usually has no digit
        base = token
    else:
        base = token[:-1]

    base = normalize_zero_initial(base)

    # longest-match for initials

    try:
        for ini in sorted(INITIALS, key=len, reverse=True):
            if base.startswith(ini):
                final = base[len(ini):]

                if ini in {"j","q","x"}:
                    final = normalize_jqx(final)
                
                return [init2idx[ini], final2idx[final], tone2idx[tone]]
    except Exception:
        raise ValueError("Invalid pinyin token")

## Char Mapping

In [5]:
import pandas as pd
import pickle as pkl


# dataset = pd.read_csv("/kaggle/input/notebooks/davidvista/pinyin-dataset-labelling/chinese-short-sentences-pinyin.csv")

# Only for Wubi and Phonetic Aware
dataset = pd.read_csv("/kaggle/input/datasets/davidvista/old-pinyin-labelled-dataset/chinese-short-sentences-pinyin.csv")


# --- Needed only for Phonetic Aware and Pure Phonetic ---
def align(token):
    if token[-1].isalpha():  # e.g. de -> de5
        return token + '5'
    return token


import pickle as pkl

with open("/kaggle/input/notebooks/davidvista/mandarin-sounds-dataset/pinyin_embeddings.pkl", "rb") as f:
    embedding_map = pkl.load(f)


dataset = dataset[
    dataset['pinyin'].str.split().transform(lambda x: all(align(y) in embedding_map.keys() for y in x))
]
# --------------------------------------------------------

In [6]:
from tqdm import tqdm 

def tokenize_dataset(dataset, tokenizer, pinyin_to_idx):

    for i, sample in tqdm(dataset.iterrows(), total=len(dataset)):
        text = sample['text']
        pinyin_str = sample['pinyin']

        char_seq = list(text)

        pinyin_tokens = pinyin_str.split()
        
        # Valid sample - yield the indices
        token_idxs, char_span = tokenizer.encode(text)

        pinyin_idxs = []
        for token in pinyin_tokens:

            try:
                pinyin_idx = pinyin_to_idx(token)
                pinyin_idxs.append(pinyin_idx)
            except ValueError:
                print(f"Skipped token {token}")
        
        yield char_seq, token_idxs, char_span, pinyin_idxs

preprocessed_dataset = tokenize_dataset(dataset, tokenizer, pinyin_to_idx)

In [7]:
char_seqs = []
token_seqs = []
char_spans = []
pinyin_seqs = []

for char_seq, token_idxs, char_span, pinyin_idxs in preprocessed_dataset:
    char_seqs.append(char_seq)
    token_seqs.append(token_idxs)
    char_spans.append(char_span)
    pinyin_seqs.append(pinyin_idxs)

100%|██████████| 161449/161449 [00:40<00:00, 3990.04it/s]


In [8]:
def build_char_vocab(char_seqs, add_pad=True, add_unk=True):
    """
    Build character vocabulary from a list of character sequences.

    Args:
        char_seqs: list of lists of strings (each inner list is a sentence of characters)
        add_pad: if True, include <PAD> token at index 0
        add_unk: if True, include <UNK> token after pad (if pad exists) or at start

    Returns:
        char2idx: dict mapping character to index
        idx2char: list mapping index to character
    """
    # Collect all unique characters
    chars = set()
    for seq in char_seqs:
        for ch in seq:
            chars.add(ch)
    # Sort for deterministic order
    sorted_chars = sorted(chars)

    # Build vocabulary list
    vocab = []
    if add_pad:
        vocab.append('<PAD>')
    if add_unk:
        vocab.append('<UNK>')
    vocab.extend(sorted_chars)

    # Create mappings
    char2idx = {ch: idx for idx, ch in enumerate(vocab)}
    idx2char = vocab

    return char2idx, idx2char

In [9]:
char2idx, idx2char = build_char_vocab(char_seqs)

# Prepare Model

## Load Model

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class ClassifierHead(nn.Module):
    """MLP with one hidden layer for classification."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class Encoder(nn.Module):
    """
    BiLSTM model for predicting pinyin from characters.
    - Token embeddings aggregated per character (EmbeddingBag).
    - Character representations passed through BiLSTM.
    - Three MLP heads predict initial, final, and tone.
    """
    def __init__(self, vocab_size_tokens, embedding_dim, hidden_dim,
                 num_initials, num_finals, num_tones,
                 num_layers=2, dropout=0.3, mlp_hidden_dim=128):
        super().__init__()
        self.embedding_bag = nn.EmbeddingBag(vocab_size_tokens, embedding_dim,
                                              mode='mean')
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            bidirectional=True, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

        # MLP heads (input = hidden_dim*2 from BiLSTM)
        self.initial_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_initials, dropout)
        self.final_head   = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_finals, dropout)
        self.tone_head    = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_tones, dropout)

    def forward(self, token_ids, token_offsets, char_lengths, return_hidden=False):
        """
        Args:
            token_ids: (total_tokens,) flattened token IDs (no padding)
            token_offsets: (total_chars,) start indices for each character
            char_lengths: (batch,) number of characters per sample
            return_hidden: if True, also return character‑level hidden states
        Returns:
            logits_initial, logits_final, logits_tone:
                each shape (batch, max_char_len, respective_vocab_size)
            (optional) char_out: (batch, max_char_len, hidden_dim*2) before heads
        """
        char_emb = self.embedding_bag(token_ids, token_offsets)  # (total_chars, emb_dim)

        # Split into per‑sample sequences and pad
        char_emb_list = torch.split(char_emb, char_lengths.tolist())
        padded_char_emb = nn.utils.rnn.pad_sequence(char_emb_list, batch_first=True)

        # Pack for LSTM
        packed_emb = pack_padded_sequence(padded_char_emb, char_lengths.cpu(),
                                          batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed_emb)
        char_out, _ = pad_packed_sequence(packed_out, batch_first=True)
        char_out = self.dropout(char_out)

        # Three MLP heads
        logits_initial = self.initial_head(char_out)
        logits_final   = self.final_head(char_out)
        logits_tone    = self.tone_head(char_out)

        if return_hidden:
            return logits_initial, logits_final, logits_tone, char_out
        return logits_initial, logits_final, logits_tone

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Decoder(nn.Module):
    """
    Decodes character‑level hidden representations (from an encoder) into character indices.
    Optionally applies a BiLSTM on top of the input hidden states.
    """
    def __init__(self, input_dim, hidden_dim, num_layers, vocab_size_char, mlp_hidden_dim=128, dropout=0.3):
        """
        Args:
            input_dim: dimension of input hidden states (e.g., encoder hidden_dim * 2)
            hidden_dim: LSTM hidden size
            num_layers: number of LSTM layers
            vocab_size_char: size of character vocabulary
            mlp_hidden_dim: hidden dimensionality of MLP classfication head
            dropout: dropout probability (applied after LSTM)
        """
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            bidirectional=True, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, vocab_size_char, dropout)

    def forward(self, hidden_states, char_lengths):
        """
        Args:
            hidden_states: (batch, max_chars, input_dim) - padded character‑level representations
            char_lengths: (batch,) - actual lengths of each sequence (for packing)
        Returns:
            logits: (batch, max_chars, vocab_size_char) - unnormalized scores for each character
        """
        # Pack for variable‑length sequences
        packed = pack_padded_sequence(
            hidden_states, char_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        lstm_out = self.dropout(lstm_out)
        logits = self.head(lstm_out)
        return logits

In [12]:
import torch
import torch.nn as nn

class EncoderDecoderPipeline(nn.Module):
    """
    Combines a character‑to‑pinyin encoder and a pinyin‑to‑character decoder.
    The encoder is expected to output character‑level hidden states (with `return_hidden=True`).
    The decoder takes those hidden states and predicts character indices.
    """
    def __init__(self, encoder, decoder, freeze_encoder=False):
        """
        Args:
            encoder: a trained/frozen encoder (e.g., CharBiLSTMForPinyin)
            decoder: a decoder (e.g., Pinyin2CharDecoder)
            freeze_encoder: if True, encoder parameters are frozen (no gradients).
        """
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, flat_tokens, char_offsets, char_lengths, return_encoder_output=False):
        """
        Args:
            flat_tokens: (total_tokens,) flattened token IDs
            char_offsets: (total_chars,) start indices for each character
            char_lengths: (batch,) number of characters per sample
        Returns:
            logits: (batch, max_chars, vocab_size_char) from decoder
        """
        # Encoder forward (returns hidden states when return_hidden=True)
        logits_initial, logits_final, logits_tone, hidden = self.encoder(flat_tokens, char_offsets, char_lengths, return_hidden=True)
        # hidden shape: (batch, max_chars, encoder_hidden_dim*2)

        # Decoder forward
        logits = self.decoder(hidden, char_lengths)

        if return_encoder_output:
            return logits_initial, logits_final, logits_tone, logits
        return logits

    def get_hidden(self, flat_tokens, char_offsets, char_lengths):
        """Return only the hidden states from the encoder (no decoder)."""
        *_, hidden = self.encoder(flat_tokens, char_offsets, char_lengths, return_hidden=True)
        return hidden

In [13]:
vocab_size_tokens = len(tokenizer.vocab)
embedding_dim = 100
hidden_dim = 512
mlp_hidden_dim = 256
num_initials = len(init2idx)
num_finals   = len(final2idx)
num_tones    = len(tone2idx)
num_layers = 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Recreate model instance
encoder = Encoder(
    vocab_size_tokens=vocab_size_tokens,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    mlp_hidden_dim=mlp_hidden_dim,
    num_initials=num_initials,
    num_finals=num_finals,
    num_tones=num_tones,
    num_layers=num_layers
).to(device)

In [14]:
input_dim = hidden_dim * 2
mlp_hidden_dim = 256
num_layers = 4
vocab_size_char = len(idx2char)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


decoder = Decoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    vocab_size_char=vocab_size_char,
    mlp_hidden_dim=mlp_hidden_dim
).to(device)

In [15]:
encoder.load_state_dict(torch.load('/kaggle/input/datasets/davidvista/phonetic-aware-encoder/encoder_best.pt', map_location=device))
decoder.load_state_dict(torch.load('/kaggle/input/datasets/davidvista/phonetic-aware-frozen-pipeline-decoder/decoder_best.pt', map_location=device))

encoder.eval()
decoder.eval()

print("Loaded models successfully!")

Loaded models successfully!


In [16]:
pipeline_frozen_encoder = EncoderDecoderPipeline(encoder, decoder, freeze_encoder=False)
pipeline_frozen_encoder.to(device)
pipeline_frozen_encoder.eval()
print("Pipeline loaded successfully.")

Pipeline loaded successfully.


## Inference

In [17]:
import torch

def predict_characters_topk(
    sentence,
    pipeline,
    tokenizer,
    idx2char,
    device,
    k=5
):
    token_ids, char_spans = tokenizer.encode(sentence)

    if len(char_spans) == 0:
        return "", []

    offsets = [span[0] for span in char_spans]

    flat_tokens = torch.tensor(
        token_ids,
        dtype=torch.long
    ).to(device)

    char_offsets = torch.tensor(
        offsets,
        dtype=torch.long
    ).to(device)

    char_lengths = torch.tensor(
        [len(char_spans)],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():

        logits = pipeline(
            flat_tokens,
            char_offsets,
            char_lengths
        )

    pred_indices = (
        logits.argmax(dim=-1)
        .squeeze(0)
        .cpu()
        .numpy()
    )

    predicted_chars = ''.join(
        idx2char[idx]
        for idx in pred_indices
    )

    topk_indices = (
        torch.topk(
            logits,
            k=min(k, logits.shape[-1]),
            dim=-1
        )
        .indices
        .squeeze(0)
        .cpu()
        .numpy()
    )

    return predicted_chars, topk_indices

In [18]:
def predict_characters(sentence, pipeline, tokenizer, idx2char, device):
    """
    Predict the character sequence for a given Chinese sentence using the trained pipeline.

    Args:
        sentence: string of Chinese characters (e.g., "你好世界")
        pipeline: trained EncoderDecoderPipeline (in eval mode)
        tokenizer: WubiCharTokenizer with encode() returning (token_ids, char_spans)
        idx2char: list mapping index -> character (including <UNK> etc.)
        device: torch device

    Returns:
        predicted_chars: string of predicted characters
    """
    # Tokenize the sentence
    token_ids, char_spans = tokenizer.encode(sentence)
    if len(char_spans) == 0:
        return ""

    # Prepare inputs
    offsets = [span[0] for span in char_spans]               # start indices of each character
    flat_tokens = torch.tensor(token_ids, dtype=torch.long).to(device)
    char_offsets = torch.tensor(offsets, dtype=torch.long).to(device)
    char_lengths = torch.tensor([len(char_spans)], dtype=torch.long).to(device)

    # Forward pass
    with torch.no_grad():
        logits = pipeline(flat_tokens, char_offsets, char_lengths)   # (1, num_chars, vocab_char)

    # Decode
    pred_indices = logits.argmax(dim=-1).squeeze(0).cpu().numpy()    # (num_chars,)
    predicted_chars = ''.join([idx2char[idx] for idx in pred_indices])

    return predicted_chars

# Benchmarking

## Load Dataset

In [19]:
perturbation_df = pd.read_csv("/kaggle/input/notebooks/davidvista/phrase-perturbation/exact_homophone_phrases.csv")

In [20]:
perturbation_df.head()

,position,original_char,replacement_char,original_word,perturbed_word,word_freq,perturbed_freq
0,0,中,忠,中国,忠国,20800,0
1,1,个,各,一个,一各,14404,0
2,0,美,每,美国,每国,11073,0
3,0,自,字,自己,字己,8939,0
4,0,可,渴,可能,渴能,8367,0


## Evaluation

**Target Outcomes**

| Metric        | Meaning                                                                              |
| ------------- | ------------------------------------------------------------------------------------ |
| Recovery Rate | Perturbed position reconstructed as the original character                           |
| Copy Rate     | Perturbed position reproduced exactly from the corrupted input                       |
| Novel Rate    | Perturbed position reconstructed as neither the original nor the perturbed character |



**Non-Target Outcomes**

| Metric        | Meaning                                      |
| ------------- | -------------------------------------------- |
| Preserve Rate | Unperturbed position reproduced correctly    |
| Change Rate   | Unperturbed position modified by the decoder |



**Phrase Outcomes**

| Metric                     | Meaning                                                                              |
| -------------------------- | ------------------------------------------------------------------------------------ |
| Full Recovery              | Prediction exactly matches the original phrase                                       |
| Full Copy                  | Prediction exactly matches the perturbed phrase                                      |
| Partial Recovery           | Perturbation corrected, but at least one unperturbed position modified               |
| Target Copy + Other Errors | Perturbation copied and at least one unperturbed position modified                   |
| Novel Target               | Perturbed position reconstructed as neither the original nor the perturbed character |


In [21]:
total = 0

# =====================================================
# Phrase-level decomposition
# =====================================================

phrase_full_recovery = 0
phrase_full_copy = 0
phrase_partial_recovery = 0
phrase_target_copy_other_errors = 0
phrase_novel_target = 0

# =====================================================
# Target position decomposition
# =====================================================

target_recovery = 0
target_copy = 0
target_novel = 0

# =====================================================
# Non-target position decomposition
# =====================================================

nontarget_preserve = 0
nontarget_change = 0

# =====================================================
# Top-k
# =====================================================

top3_correct = 0
top5_correct = 0

# =====================================================
# Examples
# =====================================================

N_EXAMPLES = 20

correct_examples = []
incorrect_examples = []

In [22]:
from tqdm import tqdm

for _, row in tqdm(
    perturbation_df.iterrows(),
    total=len(perturbation_df)
):

    original_word = row["original_word"]
    perturbed_word = row["perturbed_word"]

    prediction, topk = predict_characters_topk(
        perturbed_word,
        pipeline_frozen_encoder,
        tokenizer,
        idx2char,
        device,
        k=5
    )

    total += 1

    # =====================================================
    # Find perturbed positions
    # =====================================================

    target_positions = [
        pos
        for pos in range(len(original_word))
        if original_word[pos] != perturbed_word[pos]
    ]

    if len(target_positions) == 0:
        continue

    # =====================================================
    # Target position decomposition
    # =====================================================

    recovered_target = True

    for pos in target_positions:

        pred_char = prediction[pos] if pos < len(prediction) else None

        if pred_char == original_word[pos]:

            target_recovery += 1

        elif pred_char == perturbed_word[pos]:

            target_copy += 1
            recovered_target = False

        else:

            target_novel += 1
            recovered_target = False

    # =====================================================
    # Non-target position decomposition
    # =====================================================

    nontarget_changed = False

    for pos in range(len(original_word)):

        if pos in target_positions:
            continue

        pred_char = prediction[pos] if pos < len(prediction) else None

        if pred_char == original_word[pos]:

            nontarget_preserve += 1

        else:

            nontarget_change += 1
            nontarget_changed = True

    # =====================================================
    # Phrase-level decomposition
    # =====================================================
    
    target_states = []
    
    for pos in target_positions:
    
        pred_char = prediction[pos] if pos < len(prediction) else None
    
        if pred_char == original_word[pos]:
            target_states.append("recovery")
    
        elif pred_char == perturbed_word[pos]:
            target_states.append("copy")
    
        else:
            target_states.append("novel")
    
    all_recovered = all(
        state == "recovery"
        for state in target_states
    )
    
    all_copied = all(
        state == "copy"
        for state in target_states
    )
    
    any_novel = any(
        state == "novel"
        for state in target_states
    )
    
    if prediction == original_word:
    
        phrase_full_recovery += 1
    
        if len(correct_examples) < N_EXAMPLES:
            correct_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })
    
    elif prediction == perturbed_word:
    
        phrase_full_copy += 1
    
        if len(incorrect_examples) < N_EXAMPLES:
            incorrect_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })
    
    elif all_recovered and nontarget_changed:
    
        phrase_partial_recovery += 1
    
        if len(incorrect_examples) < N_EXAMPLES:
            incorrect_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })
    
    elif all_copied and nontarget_changed:
    
        phrase_target_copy_other_errors += 1
    
        if len(incorrect_examples) < N_EXAMPLES:
            incorrect_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })
    
    elif any_novel:
    
        phrase_novel_target += 1
    
        if len(incorrect_examples) < N_EXAMPLES:
            incorrect_examples.append({
                "original": original_word,
                "perturbed": perturbed_word,
                "prediction": prediction
            })
    
    # =====================================================
    # Top-k recovery
    # =====================================================

    recovered_top3 = True
    recovered_top5 = True

    for pos, true_char in enumerate(original_word):

        top3_chars = [
            idx2char[idx]
            for idx in topk[pos][:3]
        ]

        top5_chars = [
            idx2char[idx]
            for idx in topk[pos][:5]
        ]

        if true_char not in top3_chars:
            recovered_top3 = False

        if true_char not in top5_chars:
            recovered_top5 = False

    if recovered_top3:
        top3_correct += 1

    if recovered_top5:
        top5_correct += 1

100%|██████████| 2598/2598 [01:07<00:00, 38.23it/s]


In [23]:
import pandas as pd

# =====================================================
# Phrase-level
# =====================================================

phrase_table = pd.DataFrame({
    "Metric": [
        "Full Recovery",
        "Full Copy",
        "Partial Recovery",
        "Target Copy + Other Errors",
        "Novel Target"
    ],
    "Fraction": [
        phrase_full_recovery / total,
        phrase_full_copy / total,
        phrase_partial_recovery / total,
        phrase_target_copy_other_errors / total,
        phrase_novel_target / total
    ]
})

# =====================================================
# Target position
# =====================================================

target_total = (
    target_recovery +
    target_copy +
    target_novel
)

target_table = pd.DataFrame({
    "Metric": [
        "Recovery",
        "Copy",
        "Novel"
    ],
    "Fraction": [
        target_recovery / target_total,
        target_copy / target_total,
        target_novel / target_total
    ]
})

# =====================================================
# Non-target position
# =====================================================

nontarget_total = (
    nontarget_preserve +
    nontarget_change
)

nontarget_table = pd.DataFrame({
    "Metric": [
        "Preserve",
        "Change"
    ],
    "Fraction": [
        nontarget_preserve / nontarget_total,
        nontarget_change / nontarget_total
    ]
})

# =====================================================
# Top-k
# =====================================================

topk_table = pd.DataFrame({
    "Metric": [
        "Top-3 Recovery",
        "Top-5 Recovery"
    ],
    "Fraction": [
        top3_correct / total,
        top5_correct / total
    ]
})

print(f"Samples: {total:,}")

print("\n=== Phrase-Level Decomposition ===")
display(phrase_table)

print("\n=== Target Position Decomposition ===")
display(target_table)

print("\n=== Non-Target Position Decomposition ===")
display(nontarget_table)

print("\n=== Top-k Recovery ===")
display(topk_table)

Samples: 2,598

=== Phrase-Level Decomposition ===


,Metric,Fraction
0,Full Recovery,0.041955
1,Full Copy,0.632794
2,Partial Recovery,0.000385
3,Target Copy + Other Errors,0.092379
4,Novel Target,0.232487



=== Target Position Decomposition ===


,Metric,Fraction
0,Recovery,0.042340
1,Copy,0.725173
2,Novel,0.232487



=== Non-Target Position Decomposition ===


,Metric,Fraction
0,Preserve,0.876443
1,Change,0.123557



=== Top-k Recovery ===


,Metric,Fraction
0,Top-3 Recovery,0.320246
1,Top-5 Recovery,0.413010


In [24]:
from IPython.display import HTML, display

def color_prediction(original, prediction):

    output = []

    max_len = max(
        len(original),
        len(prediction)
    )

    for i in range(max_len):

        true_char = (
            original[i]
            if i < len(original)
            else ""
        )

        pred_char = (
            prediction[i]
            if i < len(prediction)
            else ""
        )

        if true_char == pred_char:
            color = "green"
        else:
            color = "red"

        output.append(
            f'<span style="color:{color};font-weight:bold">'
            f'{pred_char}'
            '</span>'
        )

    return ''.join(output)

In [25]:
html = "<h2>Correct Recoveries</h2>"

for ex in correct_examples:

    html += f"""
    <div style="margin-bottom:12px">
      Original:&nbsp;&nbsp;{ex['original']}<br>
      Perturbed:&nbsp;{ex['perturbed']}<br>
      Prediction:&nbsp;{color_prediction(ex['original'], ex['prediction'])}
    </div>
    """

display(HTML(html))

In [26]:
html = "<h2>Incorrect Recoveries</h2>"

for ex in incorrect_examples:

    html += f"""
    <div style="margin-bottom:12px">
      Original:&nbsp;&nbsp;{ex['original']}<br>
      Perturbed:&nbsp;{ex['perturbed']}<br>
      Prediction:&nbsp;{color_prediction(ex['original'], ex['prediction'])}
    </div>
    """

display(HTML(html))